# 🧩 Documentos para RAG

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 3 — 35 minutos**

---

## Objetivo

Entender el primer paso de RAG: cargar documentos de distintas fuentes y dividirlos en fragmentos coherentes (chunking).

## Al terminar este bloque vas a poder:

1. Cargar documentos PDF y páginas web usando LangChain.
2. Inspeccionar la estructura de un Document (page_content + metadata).
3. Explicar por qué el chunking es necesario antes de vectorizar.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **RAG** | Retrieval-Augmented Generation: buscar contexto relevante antes de generar una respuesta. |
| **Document Loader** | Componente que abre un archivo (PDF, web, etc.) y extrae su texto. |
| **Chunking** | Dividir un documento largo en fragmentos pequeños para poder vectorizarlos. |
| **Overlap** | Repetir algunos tokens del final de un chunk al inicio del siguiente para no perder contexto. |
| **page_content / metadata** | Las dos partes de todo Document en LangChain: el texto y la información contextual. |

In [ ]:
# Instalación de librerías necesarias
# LangChain es la librería principal para construir aplicaciones RAG
!uv pip install langchain -q

print("Instalación completada")

## ¿Qué es RAG y por qué empieza por los documentos?

### Analogía

Imaginá que tenés que responder un examen con libro abierto. Antes de escribir cualquier cosa, buscás en el índice, encontrás las páginas relevantes y las marcás. Eso es RAG: primero *recuperás* el contexto específico, después *generás* la respuesta fundamentada en él.

### Dónde vive esto en el mundo real

Los sistemas de atención al cliente de bancos, aseguradoras y empresas de software usan RAG para responder con información de sus manuales internos. El modelo nunca memorizó esos documentos — los busca en tiempo real. Eso significa que podés actualizar la documentación sin reentrenar nada.

### El pipeline RAG completo (tres bloques de hoy)

| Paso | Qué hace | Dónde lo vemos |
|---|---|---|
| **1. Cargar** | Extraer texto de PDFs, webs, etc. | Este notebook |
| **2. Vectorizar** | Convertir chunks en embeddings y guardarlos en ChromaDB | Siguiente |
| **3. Generar** | Recuperar contexto relevante + responder con Gemini | Último |


### ✎ Para pensar

- ¿Por qué no podemos simplemente pegarle todo el documento al LLM en lugar de fragmentarlo?
- Si un documento tiene 500 páginas, ¿cuántos chunks aproximados esperarías con chunk_size=500 caracteres?

## Carga de documentos PDF

PyPDFLoader convierte cada página del PDF en un objeto `Document` con dos campos:

- `page_content`: el texto de la página
- `metadata`: diccionario con fuente, número de página, etc.

Esta estructura uniforme es la que ChromaDB va a recibir en el siguiente notebook.

In [ ]:
import os

# Subcarpeta donde se almacenan los documentos del curso
CARPETA_DATOS = os.path.join(os.getcwd(), "CARPETA_DATOS")
print(f"Carpeta de datos: {CARPETA_DATOS}")

In [ ]:
!uv pip install pypdf -q
print("PyPDF instalado correctamente")

In [ ]:
!uv pip install -U langchain-community -q
print("langchain-community instalado correctamente")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

RUTA_PDF = os.path.join(CARPETA_DATOS, "politica_interna_tintas_acme.pdf")

loader = PyPDFLoader(RUTA_PDF)
pages = loader.load()

print(f"PDF cargado exitosamente: {len(pages)} páginas")
print("Cada página se convierte en un objeto Document independiente")

In [ ]:
# Verificamos cuántas páginas se cargaron
len(pages)

In [ ]:
# Examinamos la primera página como ejemplo
page = pages[0]
print("Primera página seleccionada para análisis")

In [ ]:
# Mostramos los primeros 500 caracteres del contenido de la página
print("CONTENIDO DE LA PRIMERA PÁGINA (primeros 500 caracteres):")
print("=" * 60)
print(page.page_content[0:500])
print("=" * 60)
print("(Contenido truncado para visualización)")

In [ ]:
# Examinamos los metadatos de la página
print("METADATOS DE LA PÁGINA:")
print("=" * 30)
print(page.metadata)
print("=" * 30)
print("Los metadatos incluyen información como número de página y archivo fuente")

## Carga desde la web

WebBaseLoader hace lo mismo con una URL: descarga el HTML, extrae el texto y lo envuelve en un `Document`.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://es.wikipedia.org/wiki/Agricultura_en_Argentina"
loader = WebBaseLoader(url)

print(f"WebBaseLoader configurado para: {url}")
print("Listo para extraer contenido web")

In [ ]:
# Cargamos el contenido de la página web
docs = loader.load()

print(f"Contenido web cargado exitosamente: {len(docs)} documento(s)")
print("El HTML ha sido procesado y convertido a texto plano")

In [ ]:
# Mostramos una muestra del contenido extraído de la web
print("CONTENIDO EXTRAÍDO DE LA WEB (primeros 1500 caracteres):")
print("=" * 60)
print(docs[0].page_content[:3000])
print("=" * 60)
print("El contenido web ha sido limpiado y estructurado para procesamiento")

In [ ]:
print("METADATOS DE LA PRIMERA PÁGINA:")
print("=" * 30)
print(docs[0].metadata)
print("=" * 30)
print("Los metadatos incluyen información como número de página y archivo fuente")

### ✎ Para pensar

- Los metadatos incluyen la fuente del documento. ¿Para qué le serviría eso al sistema RAG cuando responde?
- ¿Qué problemas podría tener cargar una página web con mucho JavaScript dinámico?

## División en fragmentos (chunking)

`RecursiveCharacterTextSplitter` divide el texto respetando jerarquía: primero párrafos, luego oraciones, luego palabras. El `chunk_overlap` repite algunos caracteres al inicio del siguiente chunk para que el contexto no se corte en seco.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

fragmentos = divisor.split_documents(docs)

print(f"✓ Documento original: {len(docs)} página(s)")
print(f"✓ Fragmentos generados: {len(fragmentos)}")
print(f"\nEjemplo — fragmento 0 ({len(fragmentos[0].page_content)} caracteres):")
print("-" * 60)
print(fragmentos[0].page_content[:300])
print("-" * 60)
print(f"\nEjemplo — fragmento 1 (primeros 100 caracteres):")
print(fragmentos[1].page_content[:100])
print("\n✎ Observá: los primeros caracteres del fragmento 1 se superponen con el final del 0.")
print("   Eso es el chunk_overlap en acción.")

### ✎ Para pensar

- Cambiá `chunk_size` a 100 y a 1000. ¿Cómo cambia la cantidad de fragmentos?
- Si `chunk_overlap` fuera 0, ¿qué información podría perderse en los bordes de los fragmentos?

### 🐛 Laboratorio de Rotura

El código de abajo *parece* razonable. Antes de ejecutarlo, predecí: ¿qué va a pasar?

```python
divisor_roto = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=200   # overlap > chunk_size
)
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Qué te dice eso sobre la relación entre `chunk_size` y `chunk_overlap`?

No mires la explicación todavía.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    divisor_roto = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=200    # ❌ overlap mayor que chunk_size
    )
    fragmentos_roto = divisor_roto.split_documents(docs)
    print("Funcionó... ¿seguro?")
except ValueError as e:
    print(f"✗ Error: {e}")
    print()
    print("LangChain no permite que chunk_overlap sea mayor que chunk_size.")
    print("Tendría que repetir MÁS texto del que tiene el fragmento — no tiene sentido.")

> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: mirá la celda de chunking y escribí, arriba de ella, un comentario de dos líneas que le explique a alguien que nunca vio esto por qué dividir el documento en pedazos es necesario antes de buscar. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

### 🐛 Laboratorio de Rotura

El código de abajo *parece* razonable. Antes de ejecutarlo, predecí: ¿qué va a pasar?

```python
divisor_roto = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=200   # overlap > chunk_size
)
```

Ejecutá la celda siguiente.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Qué te dice eso sobre la relación entre `chunk_size` y `chunk_overlap`?

No mires la explicación todavía.

### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas, para vos:

1. ¿Qué concepto de hoy te costó más encajar en la cabeza? ¿Por qué creés que se resistió?
2. Si tuvieras que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirías?

No hay respuesta correcta. Lo que escribás es el mapa de tu propio recorrido.